## Experiment 4: More Insight Slots

Full itemset pipeline but requesting ~15 insights instead of default ~8-9.
Tests H3: does slot competition prevent deep property insights from appearing?

In [1]:
from __future__ import annotations

import json
import os
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.preprocessing import TransactionEncoder
from openai import OpenAI

USER_QUERY = "Internal web app for the finance team with a relational database backend"
ARG_CACHE_PATH = Path("../../plugin/skills/azure-enterprise-infra-planner/scripts/arg_raw_output_all.json")
AOAI_BASE_URL    = os.environ.get("AZURE_OPENAI_BASE_URL", "https://workloads-assistant-aoai.openai.azure.com/openai/v1")
AOAI_DEPLOYMENT  = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-5-mini")
AOAI_API_VERSION = os.environ.get("AZURE_OPENAI_API_VERSION", "preview")

MIN_ABSOLUTE_COUNT = 4
MIN_SUPPORT_FLOOR = 0.02
MIN_ITEMSET_LEN = 2
MAX_ITEMSETS = 200
MIN_TRANSACTIONS = 30
MAX_PROPERTY_DEPTH = 5

In [2]:
_credential = DefaultAzureCredential()
_token_provider = get_bearer_token_provider(_credential, "https://cognitiveservices.azure.com/.default")

aoai = OpenAI(
    base_url=AOAI_BASE_URL,
    api_key="placeholder",
    default_query={"api-version": AOAI_API_VERSION},
    default_headers={"Authorization": f"Bearer {_token_provider()}"},
)
print(f"Azure OpenAI client ready (deployment={AOAI_DEPLOYMENT}).")

Azure OpenAI client ready (deployment=gpt-5-mini).


In [3]:
cache = Path(ARG_CACHE_PATH)
if cache.exists():
    raw = json.loads(cache.read_text(encoding="utf-8"))
    if isinstance(raw, list):
        resources = raw
    elif isinstance(raw, dict) and "data" in raw:
        resources = raw["data"]
    else:
        resources = raw
    print(f"Loaded {len(resources):,} resources from cache: {cache}")
else:
    raise FileNotFoundError(f"ARG cache not found at {cache}. Run one of the main notebooks first.")

Loaded 15,906 resources from cache: ..\plugin\skills\azure-enterprise-infra-planner\scripts\arg_raw_output_all.json


In [4]:
DROP_ARM_CHILD_TYPES = True

AUTO_CREATED_TYPES = frozenset({
    "microsoft.alertsmanagement/smartdetectoralertrules",
    "microsoft.insights/actiongroups",
    "microsoft.alertsmanagement/prometheusrulegroups",
    "microsoft.security/automations",
    "microsoft.security/pricings",
    "microsoft.operationsmanagement/solutions",
    "microsoft.security/iotsecuritysolutions",
    "microsoft.network/networkwatchers",
    "microsoft.advisor/recommendations",
})

INTERNAL_MS_RP_PREFIXES = (
    "microsoft.portalservices/",
    "microsoft.cloudtest/",
    "microsoft.hydra/",
    "microsoft.swiftlet/",
    "microsoft.compute/swiftlets",
    "microsoft.fairfieldgardens/",
    "microsoft.footprintmonitoring/",
    "microsoft.saashub/",
    "microsoft.visualstudio/",
)

AUTO_MANAGED_SUBRESOURCE_TYPES = frozenset({
    "microsoft.containerregistry/registries/replications",
    "microsoft.containerregistry/registries/webhooks",
    "microsoft.compute/capacityreservationgroups/capacityreservations",
    "microsoft.compute/hostgroups/hosts",
    "microsoft.compute/galleries/images/versions",
    "microsoft.network/networkmanagers/ipampools",
    "microsoft.network/networkmanagers/verifierworkspaces",
})

MARKETPLACE_TYPES = frozenset({
    "microsoft.solutions/applications",
    "microsoft.solutions/appliances",
    "microsoft.saas/resources",
    "microsoft.saashub/cloudservices",
})


def _is_noise(rtype: str) -> bool:
    if not rtype:
        return False
    if rtype in AUTO_CREATED_TYPES or rtype in AUTO_MANAGED_SUBRESOURCE_TYPES or rtype in MARKETPLACE_TYPES:
        return True
    if any(rtype.startswith(p) for p in INTERNAL_MS_RP_PREFIXES):
        return True
    return False


def _is_auto_created(rtype: str) -> bool:
    if not rtype:
        return False
    if DROP_ARM_CHILD_TYPES and rtype.count("/") >= 2:
        return True
    return _is_noise(rtype)


all_types = {(r.get("type") or "").lower() for r in resources if r.get("type")}
noise_types = {t for t in all_types if _is_noise(t)}
auto_created_types = {t for t in all_types if _is_auto_created(t)}
print(f"Total distinct resource types: {len(all_types)}")
print(f"Noise types: {len(noise_types)}")
print(f"Auto-created types (FIM filter): {len(auto_created_types)}")

Total distinct resource types: 182
Noise types: 32
Auto-created types (FIM filter): 51


In [5]:
PROPERTY_LEAF_WHITELIST = frozenset({
    "location", "kind",
    "sku", "name", "tier", "family", "capacity", "size",
    "publicnetworkaccess", "restrictoutboundnetworkaccess",
    "publicnetworkaccessforingestion", "publicnetworkaccessforquery",
    "defaultaction", "bypass",
    "disablelocalauth", "enablerbacauthorization",
    "minimumtlsversion", "minimaltlsversion",
    "identity",
    "keysource", "enabledoubleencryption", "enablediskencryption",
    "infrastructureencryption", "requireinfrastructureencryption",
    "zoneredundant", "zoneredundancy", "redundancymode", "replication",
    "platformfaultdomaincount",
    "backupretentionintervalinhours", "backupintervalinminutes",
    "backupstorageredundancy",
    "softdeleteretentionindays", "enablesoftdelete", "enablepurgeprotection",
    "ostype", "hypervgeneration", "licensetype",
    "accesstier", "largefilesharesstate", "allowsharedkeyaccess",
    "enablehttpstrafficonly", "supportshttpstrafficonly",
})

_KEY_DENY_RE = re.compile(
    r"(secret|password|credential|token|sas|certificate|thumbprint|fingerprint"
    r"|connection|connstr|admin(istrator)?(user|login|name)|private(ip|key|address)"
    r"|publicip|ipaddress|fqdn|hostname|host_name|endpoint|url|uri|email|mail"
    r"|principalid|tenantid|subscriptionid|objectid|clientid|appid"
    r"|customsubdomain|customdomain|key$|^key|accountkey|accesskey"
    r"|primarykey|secondarykey|sharedkey)",
    re.IGNORECASE,
)

_VALUE_DENY_PATTERNS = [
    re.compile(r"^\d{1,3}(\.\d{1,3}){3}$"),
    re.compile(r"(?i)^[0-9a-f:]{6,}$"),
    re.compile(r"(?i)^[0-9a-f]{8}-([0-9a-f]{4}-){3}[0-9a-f]{12}$"),
    re.compile(r"(?i)^https?://"),
    re.compile(r"^[^@]+@[^@]+\.[^@]+$"),
    re.compile(r"^eyJ[A-Za-z0-9_-]+\."),
    re.compile(r"^[A-Za-z0-9+/]{40,}={0,2}$"),
]


def _is_pii_key(key: str) -> bool:
    return bool(_KEY_DENY_RE.search(key))


def _is_pii_value(val: str) -> bool:
    return any(p.search(val) for p in _VALUE_DENY_PATTERNS)


def walk_properties(node: Any, path: tuple[str, ...] = (), depth: int = 0, max_depth: int = MAX_PROPERTY_DEPTH):
    if depth > max_depth:
        return
    if isinstance(node, dict):
        for k, v in node.items():
            k_lower = str(k).lower()
            if _is_pii_key(k_lower):
                continue
            yield from walk_properties(v, path + (k_lower,), depth + 1, max_depth)
    elif isinstance(node, list):
        for item in node:
            yield from walk_properties(item, path, depth + 1, max_depth)
    else:
        val_str = str(node).strip()
        if val_str and not _is_pii_value(val_str):
            yield (path, val_str)


def _insert_aggregation(tree: dict, path: tuple[str, ...], value: str) -> None:
    if not path:
        return
    leaf_name = path[-1]
    if leaf_name not in PROPERTY_LEAF_WHITELIST:
        return
    node = tree
    for segment in path[:-1]:
        node = node.setdefault(segment, {})
    counter = node.setdefault(leaf_name, Counter())
    counter[value] += 1


def aggregate_resources(rows: list[dict]) -> dict[str, dict]:
    buckets: dict[str, list[dict]] = defaultdict(list)
    for row in rows:
        rtype = (row.get("type") or "").lower()
        if not rtype or _is_noise(rtype):
            continue
        buckets[rtype].append(row)

    aggregations: dict[str, dict] = {}
    for rtype, items in sorted(buckets.items()):
        total = len(items)
        tree: dict = {}
        for item in items:
            for section in ("sku", "properties", "identity"):
                blob = item.get(section)
                if blob and isinstance(blob, dict):
                    for path, val in walk_properties(blob, (section,)):
                        _insert_aggregation(tree, path, val)
            loc = item.get("location")
            if loc:
                _insert_aggregation(tree, ("location",), str(loc).lower())
            kind = item.get("kind")
            if kind:
                _insert_aggregation(tree, ("kind",), str(kind))

        def _normalise(node, total_count):
            if isinstance(node, Counter):
                top3 = node.most_common(3)
                return {v: round(c / total_count, 3) for v, c in top3}
            if isinstance(node, dict):
                return {k: _normalise(v, total_count) for k, v in node.items()}
            return node

        prop_agg = _normalise(tree, total)
        aggregations[rtype] = {"totalCount": total, "propertyAggregations": prop_agg}

    return aggregations


resource_type_aggregations = aggregate_resources(resources)
print(f"Aggregated {len(resource_type_aggregations)} distinct resource types.")

Aggregated 150 distinct resource types.


In [6]:
def build_transactions(rows: list[dict]):
    grouped: dict[tuple[str, str], set[str]] = defaultdict(set)
    filtered: set[str] = set()
    for row in rows:
        sub = row.get("subscriptionId")
        rg = (row.get("resourceGroup") or "").lower()
        rtype = (row.get("type") or "").lower()
        if not sub or not rg or not rtype:
            continue
        if _is_auto_created(rtype):
            filtered.add(rtype)
            continue
        grouped[(sub, rg)].add(rtype)

    raw_rg_count = sum(1 for items in grouped.values() if items)
    transactions = [sorted(items) for items in grouped.values() if len(items) >= 2]
    single_type_rg_count = raw_rg_count - len(transactions)
    return transactions, sorted(filtered), raw_rg_count, single_type_rg_count


def _reliability(count: int) -> str:
    if count >= 30:
        return "high"
    if count >= 10:
        return "medium"
    if count >= 5:
        return "low"
    return "exploratory"


def compute_mfis(itemsets: list[dict]) -> list[dict]:
    sets = [(frozenset(it["itemset"]), it) for it in itemsets]
    sets.sort(key=lambda s: (-len(s[0]), -float(s[1].get("support", 0))))
    kept: list[tuple[frozenset, dict]] = []
    for s, entry in sets:
        if any(s < ks for ks, _ in kept):
            continue
        kept.append((s, entry))
    return [entry for _, entry in kept]


transactions, filtered_types, raw_rg_count, single_type_rg_count = build_transactions(resources)
print(f"Transactions: {len(transactions)}")

n = len(transactions)
mfis: list[dict] = []
min_support = 0.0

if n > 0:
    min_support = max(MIN_SUPPORT_FLOOR, MIN_ABSOLUTE_COUNT / n)
    te = TransactionEncoder()
    te_ary = te.fit_transform(transactions)
    df = pd.DataFrame(te_ary, columns=te.columns_)
    freq_df = fpgrowth(df, min_support=min_support, use_colnames=True)
    freq_df = freq_df[freq_df["itemsets"].apply(len) >= MIN_ITEMSET_LEN]

    if len(freq_df) > 0:
        all_itemsets = []
        for _, row in freq_df.iterrows():
            sup = float(row["support"].item()) if hasattr(row["support"], "item") else float(row["support"])
            count = int(round(sup * n))
            all_itemsets.append({
                "itemset": sorted(row["itemsets"]),
                "support": round(sup, 4),
                "count": count,
                "reliability": _reliability(count),
            })
        mfis = compute_mfis(all_itemsets)
        if len(mfis) > MAX_ITEMSETS:
            mfis = sorted(mfis, key=lambda x: (-x["count"], -len(x["itemset"])))[:MAX_ITEMSETS]

    print(f"Maximal Frequent Itemsets: {len(mfis)}")
else:
    print("No transactions available.")

Transactions: 490
Maximal Frequent Itemsets: 76


In [7]:
sub_count = len({r.get("subscriptionId") for r in resources if r.get("subscriptionId")})
rg_count = len({
    (r.get("subscriptionId"), (r.get("resourceGroup") or "").lower())
    for r in resources
    if r.get("subscriptionId") and r.get("resourceGroup")
})

payload = {
    "userQuery": USER_QUERY,
    "resourceContext": {
        "subscriptionCount": sub_count,
        "resourceGroupCount": rg_count,
        "resourceTypes": resource_type_aggregations,
    },
    "tenantPatterns": {
        "transactionUnit": "resourceGroup",
        "algorithm": "fpgrowth",
        "minSupport": round(min_support, 4),
        "reliabilityBuckets": {
            "high": ">=30 RGs",
            "medium": ">=10 RGs",
            "low": ">=5 RGs",
            "exploratory": f">={MIN_ABSOLUTE_COUNT} RGs",
        },
        "maximalFrequentItemsets": mfis,
    },
}

payload_json = json.dumps(payload, ensure_ascii=False)
print(f"Payload size: {len(payload_json):,} chars")
print(f"MFIs included: {len(mfis)}")

Payload size: 65,558 chars
MFIs included: 76


In [8]:
# KEY EXPERIMENT: Same itemset prompt but requesting approximately 15 insights
INSIGHTS_PROMPT = """# Role and Objective
You are an expert Azure Insight Agent. Your mission is to analyze the user's existing infrastructure data and produce insights that inform downstream infrastructure plan generation.

# Input Data Format

You will receive a JSON object with three sections:

## userQuery
The user's infrastructure request. Focus your insights on patterns most relevant to this request, but draw from all tenant-wide data.

## resourceContext
Per-resource-type property aggregations across the tenant.
- Each key is an ARM resource type (e.g. "microsoft.storage/storageaccounts").
- `totalCount`: how many instances of this type exist in the tenant.
- `propertyAggregations`: nested object where each leaf is a dict of `{value: fraction}`.
  - `fraction` is the share of instances that have that value (0.0\u20131.0). The top 3 values are shown; the implied remainder is `1 - sum(fractions)`.
  - Example: `"location": {"eastus": 0.6, "westus2": 0.3}` means 60% of instances are in eastus, 30% in westus2, and 10% elsewhere.

## tenantPatterns
Frequent co-occurrence patterns mined via FP-Growth across all resource groups.
- `maximalFrequentItemsets`: list of Maximal Frequent Itemsets (MFIs).
  - Each MFI is a set of ARM resource types that frequently appear together in the same resource group.
  - `itemset`: the list of ARM types in the pattern.
  - `support`: fraction of resource groups (0.0\u20131.0) that contain ALL types in this itemset.
  - `reliability`: quality tier based on absolute count \u2014 "high" (\u226530 RGs), "medium" (\u226510), "low" (\u22655), "exploratory" (\u22654).
  - An MFI is \"maximal\" because no larger frequent itemset contains it as a subset. Every subset of an MFI is also frequent.
  - Example: `{"itemset": ["microsoft.keyvault/vaults", "microsoft.storage/storageaccounts", "microsoft.network/virtualnetworks"], "support": 0.15, "reliability": "medium"}` means 15% of resource groups contain all three of these types together.

# Process
1. Analyze the property aggregations to identify architectural conventions (SKU choices, security posture, region preferences, redundancy settings).
2. Analyze the MFIs to identify common resource grouping patterns \u2014 which services the tenant typically deploys together.
3. Cross-reference both: e.g., if an MFI shows KeyVault+Storage co-deploy often, and property aggregations show Storage uses LRS 80% of the time, combine into a richer insight.
4. Focus on patterns relevant to the user's query, but include important tenant-wide conventions too.
5. Re-examine your insights for completeness and accuracy.

# Insight Guidelines
When selecting resource properties to base insights on:
- Only consider properties that represent explicit user decisions affecting design.
- Never include properties involving runtime, versions, implementation details, app settings, default values, operational settings, or boilerplate configurations.
- Never include instance-specific properties of a resource.

### Structure of an Insight

Each insight must contain three parts: an observed pattern, the reasoning behind it, and a planning implication.
- The reasoning must be grounded in factual information from the data. Do not make assumptions.
- The planning implication must state concrete actions or decisions for infra planning that align with the user's requirements.
- The reasoning must clearly connect the observed pattern to the planning implication.

### Filtering

Use the following areas as a guide when deciding which resource properties are meaningful:
- Region
- Resource pairing
- Security posture
- Cost
- Naming and tagging conventions
- Azure policies

# Output

Produce approximately 15 insights to ensure comprehensive coverage of both property-level conventions and co-occurrence patterns.

Return a JSON object with an \"insights\" key containing an array of insight strings.

```json
{
  "insights": [
    "Insight 1",
    "Insight 2",
    "..."
  ]
}
```

Each insight must be a single sentence with this structure: \"[observed pattern]: [reasoning] [planning implication]\"."""


print(f"Sending payload ({len(payload_json):,} chars) to Azure OpenAI ...")

response = aoai.chat.completions.create(
    model=AOAI_DEPLOYMENT,
    messages=[
        {"role": "system", "content": INSIGHTS_PROMPT},
        {"role": "user", "content": payload_json},
    ],
    response_format={"type": "json_object"},
)

raw_text = response.choices[0].message.content or "{}"
print(f"Response received ({len(raw_text):,} chars).")

parsed = json.loads(raw_text)

insights: list[str] = []
if isinstance(parsed, list):
    insights = [str(x) for x in parsed]
elif isinstance(parsed, dict):
    for key in ("insights", "Insights", "items", "results"):
        if key in parsed and isinstance(parsed[key], list):
            insights = [str(x) for x in parsed[key]]
            break
    if not insights:
        for v in parsed.values():
            if isinstance(v, list) and v and isinstance(v[0], str):
                insights = [str(x) for x in v]
                break

print(f"\nExtracted {len(insights)} insights.")

Sending payload (65,558 chars) to Azure OpenAI ...
Response received (5,416 chars).

Extracted 15 insights.


In [9]:
output_path = Path("results/more_insights.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(insights, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Wrote {len(insights)} insights to {output_path}")
print()
for i, insight in enumerate(insights, 1):
    print(f"{i:2}. {insight}")

Wrote 15 insights to results\exp4_more_insights.json

 1. KeyVaults are predominantly deployed with public network access enabled and soft-delete configured (most with 90-day retention): because KeyVaults are currently reachable over public endpoints but already use soft-delete, plan to harden secret key storage for the finance app by moving KeyVaults to private endpoints or service endpoints and enabling purge protection to prevent accidental or malicious deletion.
 2. User-assigned managed identities are widespread and frequently co-deployed with KeyVault and application telemetry in resource-group patterns: since managed identities are already a common access model, plan to use a user-assigned managed identity for the finance web app to access KeyVault and the database to avoid embedding credentials.
 3. Network Security Groups and Virtual Networks are routinely present alongside web apps and public IPs in frequent itemsets: because the tenant consistently places app tiers inside VN